# 🔬 DermAI — Entrenamiento de Modelos Deep Learning en Google Colab

Este cuaderno interactivo de Jupyter/Colab está diseñado para entrenar los **5 modelos de Deep Learning** utilizados en la aplicación **DermAI** sobre el dataset **ISIC 2019** (clasificación binaria benigno/maligno):

1. **DenseNet121**
2. **EfficientNetB0**
3. **ResNet50**
4. **InceptionV3**
5. **MobileNetV2**

## Requisitos previos:
1. Activar el entorno de ejecución con **GPU** en Colab: `Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU` (o superior).

## 1. Verificar la GPU
Ejecuta la siguiente celda para asegurarte de que tienes una GPU asignada. Esto acelerará el entrenamiento significativamente.

In [ ]:
import tensorflow as tf
print("Versión de TensorFlow:", tf.__version__)
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print("✅ GPU detectada:", gpu_devices)
    !nvidia-smi
else:
    print("⚠️ No se detectó ninguna GPU. Ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno de ejecución' y selecciona una GPU (como T4 GPU) antes de continuar.")

## 2. Configurar API de Kaggle y descargar ISIC 2019
Esta celda configura automáticamente tu **Token de la API de Kaggle** descargando el dataset sin necesidad de subir archivos manuales.

In [ ]:
import os
import zipfile

# Configurar Token de la API de Kaggle
KAGGLE_TOKEN = "KGAT_4a8b25ff40caeda1ef26d9ebd1938db4"

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

access_token_path = os.path.join(kaggle_dir, 'access_token')
with open(access_token_path, 'w') as f:
    f.write(KAGGLE_TOKEN)
os.chmod(access_token_path, 0o600)
print("✅ Token de la API de Kaggle configurado con éxito.")

# Descargar el dataset ISIC 2019 (andrewmvd/isic-2019)
if not os.path.exists("ISIC_2019_Training_GroundTruth.csv"):
    print("⏳ Descargando dataset de Kaggle (esto puede tardar unos minutos)...")
    !kaggle datasets download -d andrewmvd/isic-2019 -p .
    
    # Buscar el zip
    zip_path = None
    for file in os.listdir("."):
        if file.endswith(".zip") and "isic" in file.lower():
            zip_path = file
            break
            
    if zip_path:
        print(f"📦 Extrayendo {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(".")
        print("✅ Dataset extraído con éxito.")
    else:
        print("❌ No se encontró el archivo zip descargado.")
else:
    print("✅ El dataset ya se encuentra descargado y extraído en el entorno.")

## 3. Preprocesar las etiquetas e imágenes
En esta sección cargaremos la tabla CSV de etiquetas y re-etiquetaremos los datos a binario:
- **Maligno (1)**: Melanoma (MEL), Basal Cell Carcinoma (BCC), Actinic Keratosis (AK), Squamous Cell Carcinoma (SCC).
- **Benigno (0)**: Melanocytic Nevus (NV), Benign Keratosis (BKL), Dermatofibroma (DF), Vascular Lesion (VASC), Unknown (UNK).

También dividiremos el dataset en splits del **70% entrenamiento**, **10% validación** y **20% prueba** manteniendo el balance de clases.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("ISIC_2019_Training_GroundTruth.csv")

# Agrupación binaria
malignant_cols = ['MEL', 'BCC', 'AK', 'SCC']
df['is_malignant'] = df[malignant_cols].max(axis=1).astype(int)

# Identificar la carpeta correcta de imágenes
img_dir_candidates = [
    "ISIC_2019_Training_Input/ISIC_2019_Training_Input",
    "ISIC_2019_Training_Input",
    "isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input"
]
base_img_dir = None
for candidate in img_dir_candidates:
    if os.path.exists(candidate):
        base_img_dir = candidate
        break

if base_img_dir is None:
    raise FileNotFoundError("No se encontró el directorio de imágenes ISIC_2019_Training_Input")

print(f"📂 Carpeta de imágenes en uso: {base_img_dir}")
df['image_path'] = df['image'].apply(lambda x: os.path.join(base_img_dir, f"{x}.jpg"))

# Verificar existencia de imágenes
print("⏳ Verificando archivos físicos en disco...")
df['exists'] = df['image_path'].apply(os.path.exists)
df = df[df['exists'] == True].reset_index(drop=True)
df['label_str'] = df['is_malignant'].astype(str)
print(f"✅ Imágenes válidas encontradas: {len(df)}")

# Divisiones estratificadas
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['is_malignant']
)
val_df, test_df = train_test_split(
    temp_df, test_size=2/3, random_state=42, stratify=temp_df['is_malignant']
)

print(f"📊 Resumen de splits:")
print(f"  • Entrenamiento: {len(train_df)} imágenes ({train_df['is_malignant'].mean()*100:.1f}% Maligno)")
print(f"  • Validación:   {len(val_df)} imágenes ({val_df['is_malignant'].mean()*100:.1f}% Maligno)")
print(f"  • Test:         {len(test_df)} imágenes ({test_df['is_malignant'].mean()*100:.1f}% Maligno)")

## 4. Crear los generadores de datos con Data Augmentation
Utilizaremos `ImageDataGenerator` para realizar transformaciones aleatorias (rotaciones, zoom, volteo) en el conjunto de entrenamiento, mejorando la generalización de los modelos. Los conjuntos de validación y test solo se normalizan.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Data Augmentation para entrenamiento
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

# Solo reescalado para validación y test
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_path',
    y_col='label_str',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='image_path',
    y_col='label_str',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='image_path',
    y_col='label_str',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

## 5. Definir la función de construcción del modelo (Transfer Learning)
Usaremos modelos base pre-entrenados en ImageNet y agregaremos una cabeza clasificadora adaptada. Descongelaremos las últimas capas del modelo base para un ajuste fino (Fine-tuning).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

LEARNING_RATE = 1e-4

def build_model(model_name, base_arch_func):
    print(f"🏗️ Construyendo {model_name}...")
    
    # Cargar modelo base con pesos de ImageNet
    base_model = base_arch_func(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    
    # Dejar entrenable las últimas 20 capas para fine-tuning
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
        
    # Definir entradas y preprocesamiento específico por modelo
    inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = inputs
    
    # Preprocesamiento específico para coincidir con el entrenamiento original
    if "EfficientNet" in model_name:
        pass
    elif "ResNet" in model_name:
        x = layers.Lambda(lambda v: v * 255.0)(x)
        x = keras.applications.resnet50.preprocess_input(x)
    elif "Inception" in model_name:
        x = layers.Lambda(lambda v: (v - 0.5) * 2.0)(x)
    elif "DenseNet" in model_name:
        x = layers.Lambda(lambda v: v * 255.0)(x)
        x = keras.applications.densenet.preprocess_input(x)
    elif "MobileNet" in model_name:
        x = layers.Lambda(lambda v: (v - 0.5) * 2.0)(x)
        
    x = base_model(x, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid', name='output')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

## 6. Bucle de Entrenamiento de todos los modelos
En esta celda definimos las arquitecturas a entrenar y lanzamos el bucle. Se implementan callbacks de `EarlyStopping` (para detener el entrenamiento si deja de mejorar) y `ReduceLROnPlateau` (para reducir la tasa de aprendizaje si el progreso se estanca).

In [ ]:
import os
from sklearn.metrics import confusion_matrix, matthews_corrcoef

MODEL_ARCHITECTURES = {
    "EfficientNetB0": keras.applications.EfficientNetB0,
    "ResNet50": keras.applications.ResNet50,
    "MobileNetV2": keras.applications.MobileNetV2,
    "DenseNet121": keras.applications.DenseNet121,
    "InceptionV3": keras.applications.InceptionV3,
}

output_dir = "models_output"
os.makedirs(output_dir, exist_ok=True)
resultados_totales = {}

EPOCHS = 20  # Aumentar a 40-50 para obtener mejores resultados reales

for model_name, arch_func in MODEL_ARCHITECTURES.items():
    print(f"\n=======================================================")
    print(f"🔥 ENTRENANDO MODELO: {model_name}")
    print(f"=======================================================")
    
    model = build_model(model_name, arch_func)
    model_save_path = os.path.join(output_dir, f"{model_name.lower()}_isic2019.keras")
    
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1),
        keras.callbacks.ModelCheckpoint(filepath=model_save_path, monitor='val_auc', mode='max', save_best_only=True, verbose=1)
    ]
    
    # Ajustar modelo
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=val_generator,
        callbacks=callbacks,
        verbose=1
    )
    
    print(f"✅ Entrenamiento de {model_name} completado.")
    
    # Evaluación del modelo en Test
    print(f"⏳ Evaluando {model_name} en el conjunto de test...")
    test_generator.reset()
    y_pred_probs = model.predict(test_generator, verbose=1).flatten()
    y_true = test_generator.classes
    
    # Calcular métricas
    y_pred = (y_pred_probs >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_score = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    mcc = matthews_corrcoef(y_true, y_pred)
    
    resultados_totales[model_name] = {
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1_score": f1_score,
        "mcc": mcc,
        "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]]
    }
    
    print(f"📈 Métricas en Test para {model_name}:")
    print(f"  • Accuracy:    {accuracy*100:.2f}%")
    print(f"  • Sensibilidad: {sensitivity*100:.2f}%")
    print(f"  • Especificidad:{specificity*100:.2f}%")
    print(f"  • MCC:          {mcc:.4f}")

## 7. Resultados finales e Integración en config.py
Ejecuta esta celda para generar el fragmento de código de configuración que puedes pegar directamente en tu archivo `config.py` local de la aplicación DermAI.

In [ ]:
print("📋 Copia el siguiente código y reemplázalo en tu archivo 'config.py':\n")
print("REAL_TRAINING_METRICS = {")
for m_name, metrics in resultados_totales.items():
    print(f"    '{m_name}': {{")
    print(f"        'accuracy': {metrics['accuracy']:.4f},")
    print(f"        'sensitivity': {metrics['sensitivity']:.4f},")
    print(f"        'specificity': {metrics['specificity']:.4f},")
    print(f"        'precision': {metrics['precision']:.4f},")
    print(f"        'f1_score': {metrics['f1_score']:.4f},")
    print(f"        'mcc': {metrics['mcc']:.4f},")
    print(f"        'confusion_matrix': {metrics['confusion_matrix']},")
    print(f"    }},")
print("}")

print("\n📦 MODELOS DISPONIBLES EN EL ENTORNO:")
for file in os.listdir(output_dir):
    print(f"  - {os.path.join(output_dir, file)}")

## 8. Guardar/Descargar Modelos
Puedes descargar los archivos `.keras` a tu computadora y moverlos a la carpeta `app/models/` de tu proyecto local, o montando Google Drive para guardarlos de forma permanente.

In [ ]:
# Opción 1: Comprimir y descargar directamente en tu navegador
import shutil
from google.colab import files

shutil.make_archive('modelos_entrenados_dermai', 'zip', 'models_output')
print("📦 Archivo comprimido creado: 'modelos_entrenados_dermai.zip'")
print("⏳ Iniciando descarga automática... (Si se bloquea, descárgalo desde la barra lateral izquierda en Archivos)")
try:
    files.download('modelos_entrenados_dermai.zip')
except Exception as e:
    print("Descarga no iniciada de forma automática:", e)
